# Ablation figures

Produces the two ablation figures used in the paper (`figures/shipgroup_modules_{ade,k_ade}.pdf`,
`figures/ablation_modules_fde.pdf`) from the committed `lightning_logs/**/eval_fh.csv` +
`hparams.yaml` files -- no raw TensorBoard event logs needed (those aren't shipped in
this repo; see the README).

This is a trimmed-down version of the exploratory notebook used during development --
only the cells that produced the final figures are kept.

In [ ]:
import pandas as pd
import plotly.graph_objs as go
import plotly.express as px
import numpy as np
from plotly.colors import hex_to_rgb
from plotly.subplots import make_subplots
from pathlib import Path
import yaml

pd.options.plotting.backend = "plotly"

In [ ]:
MODULE_COLS = ["nereus_modules.social", "nereus_modules.map", "nereus_modules.prior"]
MODULE_TITLES = {
    "nereus_modules.social": "Social",
    "nereus_modules.map": "Map",
    "nereus_modules.prior": "Prior",
}

SCALAR_COLS = ["ade", "k_ade", "pred_risk", "min_pred_dist", "collision_ratio"]

METRIC_LABELS = {
    "ade": "ADE [m]",
    "k_ade": "minADE [m]",
    "fde": "FDE [m]",
    "k_fde": "minFDE [m]",
}
PAPER_COLORS = px.colors.qualitative.Set2
SHIP_GROUPS = ["all", "cargo", "passenger", "sailing", "other"]

In [ ]:
def _rgba(c, a):
    """Plotly color string ('#rrggbb' or 'rgb(r,g,b)') -> 'rgba(r,g,b,a)'."""
    if c.startswith("#"):
        r, g, b = hex_to_rgb(c)
    else:
        r, g, b = (int(x) for x in c[c.find("(") + 1:c.find(")")].split(",")[:3])
    return f"rgba({r},{g},{b},{a})"


def _setting_label(series):
    """Map a module hyperparameter column to clean on/off-style labels ('off' when disabled)."""
    s = series.where(series.notna(), "off")
    return s.replace({None: "off", "null": "off", "None": "off"}).astype(str)


def build_setting_colors(df_hp, modules=MODULE_COLS, palette=PAPER_COLORS):
    """Fixed color per setting value across every module/plot (e.g. 'off' is always one color)."""
    labels = sorted(pd.unique(pd.concat([_setting_label(df_hp[m]) for m in modules])))
    return {lab: palette[i % len(palette)] for i, lab in enumerate(labels)}


def _setting_color(setting):
    """Color for a setting, extending SETTING_COLORS deterministically for unseen values."""
    if setting not in SETTING_COLORS:
        SETTING_COLORS[setting] = PAPER_COLORS[len(SETTING_COLORS) % len(PAPER_COLORS)]
    return SETTING_COLORS[setting]


def module_by_shipgroup(long, df_hp, module_col, metric="ade", region="kiel"):
    """Mean metric per (ship_group, module setting), averaged over models & horizon."""
    df = long[long["region"] == region]
    per_model = df.groupby(["dir_name", "ship_group"])[metric].mean().reset_index()
    setting = df_hp[module_col].replace({None: "off", "null": "off", "None": "off"})
    setting = setting.where(setting.notna(), "off")
    per_model[module_col] = per_model["dir_name"].map(setting)
    return per_model


def load_eval_csvs(log_dir, source="fh"):
    """Concatenate every per-model full_eval CSV, tagged by version (`dir_name`)."""
    frames = []
    for csv_path in sorted(Path(log_dir).glob(f"version_*/eval_{source}.csv")):
        df = pd.read_csv(csv_path)
        df["dir_name"] = csv_path.parent.name
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No eval_{source}.csv files under {log_dir}")
    return pd.concat(frames, ignore_index=True)




def eval_scalar_metrics(long, region=None, ship_group="all"):
    """One row per model of the horizon-independent scalars (ade, collision, …)."""
    df = long[long["ship_group"] == ship_group]
    if region is not None:
        df = df[df["region"] == region]
    return df.groupby("dir_name")[SCALAR_COLS].mean()


def load_hp_from_yaml(log_dir, columns=None):
    """Build the df_hp table (one row per run, indexed by dir_name) directly from each
    run's hparams.yaml -- the raw-tfevents-based tbparse loader isn't usable here since
    this repo only ships hparams.yaml/eval_fh.csv/loss_curve.csv, not raw event logs."""
    rows = {}
    for hp_path in sorted(Path(log_dir).glob("version_*/hparams.yaml")):
        with open(hp_path) as f:
            hp = yaml.safe_load(f) or {}
        rows[hp_path.parent.name] = hp
    df_hp = pd.DataFrame(rows).T
    df_hp["v_name"] = df_hp.index
    if columns is None:
        return df_hp
    return df_hp[columns]

In [ ]:
log_dir = "lightning_logs/nereus_ablation/"
df_hp = load_hp_from_yaml(log_dir)
SETTING_COLORS = build_setting_colors(df_hp)
eval_long = load_eval_csvs(log_dir, source="fh")
print("models evaluated:", eval_long["dir_name"].nunique())

In [ ]:



df_eval_scalar = eval_scalar_metrics(eval_long, region="kiel", ship_group="all")
df_eval_scalar = df_eval_scalar.join(df_hp[MODULE_COLS])
df_eval_scalar = df_eval_scalar.fillna("off")

In [ ]:
import plotly.express as px

x="nereus_modules.prior"#"nereus_modules.social"
y="nereus_modules.map"
z="ade"
color = "nereus_modules.social" #ade

fig = px.scatter_3d(
    df_eval_scalar,
    x=x,
    y=y,
    z=z,
    color=color,
    color_continuous_scale="Viridis",
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(
    scene=dict(
        xaxis_title=x,
        yaxis_title=y,
        zaxis_title=z,
    )
)
fig.show()


## Figure: `shipgroup_modules_{ade,k_ade}.pdf`

Mean metric per ship group, one box per module setting, one subplot per module.

In [ ]:
Path("figures").mkdir(exist_ok=True)

for PLOT_METRIC in ["ade", "k_ade"]:
    per_module_data = {m: module_by_shipgroup(eval_long, df_hp, m, metric=PLOT_METRIC)
                       for m in MODULE_COLS}
    all_settings = sorted(
        {s for m, pm in per_module_data.items() for s in pm[m].unique()},
        key=lambda x: (x != "off", x),
    )
    setting_color = {s: PAPER_COLORS[i % len(PAPER_COLORS)] for i, s in enumerate(all_settings)}

    fig = make_subplots(
        rows=1, cols=3, shared_yaxes=True,
        subplot_titles=[MODULE_TITLES[m] for m in MODULE_COLS],
        horizontal_spacing=0.04,
    )

    for i, module_col in enumerate(MODULE_COLS, start=1):
        pm = per_module_data[module_col]
        settings = sorted(pm[module_col].unique(), key=lambda x: (x != "off", x))

        legend_key = "legend" if i == 1 else f"legend{i}"
        for s in settings:
            sub = pm[pm[module_col] == s]
            fig.add_trace(
                go.Box(
                    x=sub["ship_group"], y=sub[PLOT_METRIC],
                    name=s,
                    marker_color=setting_color[s],
                    offsetgroup=s,
                    legend=legend_key,
                    legendgroup=f"{module_col}/{s}",
                ),
                row=1, col=i,
            )
        fig.update_xaxes(categoryorder="array", categoryarray=SHIP_GROUPS, row=1, col=i)

    for i, module_col in enumerate(MODULE_COLS, start=1):
        legend_key = "legend" if i == 1 else f"legend{i}"
        axis_key = "xaxis" if i == 1 else f"xaxis{i}"
        x0, x1 = fig.layout[axis_key].domain
        fig.update_layout({legend_key: dict(
            orientation="h",
            x=(x0 + x1) / 2, xanchor="center",
            y=-0.15, yanchor="top",
        )})

    fig.update_layout(
        boxmode="group",
        width=1100, height=420,
        margin=dict(t=40, b=80),
    )
    fig.update_yaxes(title_text=METRIC_LABELS.get(PLOT_METRIC, PLOT_METRIC), row=1, col=1)
    fig.show()
    fig.write_image(f"figures/shipgroup_modules_{PLOT_METRIC}.pdf")

## Figure: `ablation_modules_fde.pdf`

FDE vs. prediction horizon, one line per module setting, one subplot per module.

In [ ]:
PLOT_METRIC_STEP = "fde"   # matches the figure actually used in the paper
SHOW_BANDS = False          # ±1 std band over runs with the same setting

fig = make_subplots(
    rows=1, cols=3, shared_yaxes=True,
    subplot_titles=[MODULE_TITLES[m] for m in MODULE_COLS],
    horizontal_spacing=0.04,
)

df = eval_long[(eval_long["ship_group"] == "all") & (eval_long["region"] == "kiel")]

for i, module_col in enumerate(MODULE_COLS, start=1):
    sub = df.copy()
    sub["setting"] = _setting_label(sub["dir_name"].map(df_hp[module_col]))
    stats = (sub.groupby(["setting", "minute"])[PLOT_METRIC_STEP]
                .agg(mean="mean", std="std").reset_index().sort_values("minute"))

    legend_key = "legend" if i == 1 else f"legend{i}"
    settings = sorted(stats["setting"].unique(), key=lambda x: (x != "off", x))
    for setting in settings:
        sdf = stats[stats["setting"] == setting]
        color = _setting_color(setting)
        x, y, e = sdf["minute"].to_numpy(), sdf["mean"].to_numpy(), sdf["std"].fillna(0).to_numpy()
        if SHOW_BANDS:
            fig.add_trace(go.Scatter(
                x=np.concatenate([x, x[::-1]]), y=np.concatenate([y + e, (y - e)[::-1]]),
                fill="toself", fillcolor=_rgba(color, 0.15), line=dict(width=0),
                hoverinfo="skip", showlegend=False,
                legendgroup=f"{module_col}/{setting}",
            ), row=1, col=i)
        fig.add_trace(go.Scatter(
            x=x, y=y, mode="lines", name=setting,
            line=dict(color=color, width=2.5),
            legend=legend_key, legendgroup=f"{module_col}/{setting}",
        ), row=1, col=i)
    fig.update_xaxes(title_text="Prediction horizon [min]", row=1, col=i)

for i, module_col in enumerate(MODULE_COLS, start=1):
    legend_key = "legend" if i == 1 else f"legend{i}"
    axis_key = "xaxis" if i == 1 else f"xaxis{i}"
    x0, x1 = fig.layout[axis_key].domain
    fig.update_layout({legend_key: dict(
        orientation="h",
        x=(x0 + x1) / 2, xanchor="center",
        y=-0.22, yanchor="top",
    )})

fig.update_layout(
    template="simple_white",
    width=1100, height=420, font=dict(size=15),
    margin=dict(t=40, b=95),
)
fig.update_yaxes(title_text=METRIC_LABELS.get(PLOT_METRIC_STEP, PLOT_METRIC_STEP), row=1, col=1)
fig.show()
fig.write_image(f"figures/ablation_modules_{PLOT_METRIC_STEP}.pdf")